In [ ]:
from src.imports import DATA_DIR


In [8]:
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer
from emoji import demojize


In [5]:
df = pd.read_csv(DATA_DIR / 'tweets_brutos_bronze2.csv', index_col = 0)

In [6]:
df.head()

,full_text
0,se o lula ganhar eu quero uma roda de beijo co...
1,@ixslorena LULA PRESIDENTE HOJE
2,mãe morrendo de alegria na carreata do Lula
3,@trajaza AVISA QUE MEU PAI LULA VAI GANHAR PRI...
4,@indieoffw O RJ elegendo castro e juram que va...


In [7]:
len(df)

270301

In [9]:
tokenizer = AutoTokenizer.from_pretrained(
    "melll-uff/bertweetbr",
    normalization=True
)

tokenizer.demojizer = lambda x: demojize(x, language="pt")


In [10]:
df["full_text"] = df["full_text"].astype("string")

In [ ]:
def normalizeTweet(text):
    if pd.isna(text):
        return pd.NA
    return tokenizer.normalizeTweet(text)

In [ ]:
tqdm.pandas(desc="Normalizando tweets")
df["clean_text"] = df["full_text"].progress_apply(normalizeTweet)

Normalizando tweets: 100%|██████████| 270301/270301 [00:35<00:00, 7685.22it/s]


In [ ]:
i = 1234
assert normalizeTweet(df.loc[i, "full_text"]) == normalizeTweet(df.loc[i, "full_text"])

In [14]:
df = (
    df
    .dropna(subset=["clean_text"])
    .drop_duplicates(subset="clean_text", keep="first")
    .reset_index(drop=True)
)

print(len(df))

192954


In [17]:
print(len(df))
print(df["clean_text"].isna().sum())
print(df["clean_text"].duplicated().sum())


192954
0
0


In [18]:
df.head()

,full_text,clean_text
0,se o lula ganhar eu quero uma roda de beijo co...,se o lula ganhar eu quero uma roda de beijo co...
1,@ixslorena LULA PRESIDENTE HOJE,@USER LULA PRESIDENTE HOJE
2,mãe morrendo de alegria na carreata do Lula,mãe morrendo de alegria na carreata do Lula
3,@trajaza AVISA QUE MEU PAI LULA VAI GANHAR PRI...,@USER AVISA QUE MEU PAI LULA VAI GANHAR PRIMEI...
4,@indieoffw O RJ elegendo castro e juram que va...,@USER O RJ elegendo castro e juram que vai dar...


In [ ]:
df.to_parquet(
    "tweets_base_normalizados.parquet",
    index=False
)